[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/05_store-recommender.ipynb)

*Industry Project 5 of 10 · ML Engineer · Intermediate · about 7–9 hours*

# "You Might Also Like": A Store Recommender

**The situation.** *ShopKart* wants a "You might also like" row on every customer's home page. You have a year of purchase history: 400 customers, 150 products. The product manager asks:

1. Can we beat simply showing the bestsellers?
2. What do we show a brand-new product that nobody has bought yet?
3. What do we show a customer who has bought almost nothing?
4. Why was a product recommended (so we can show "Because you bought …")?

You'll build each piece and measure it the way recommendation teams do: hide each customer's latest purchase and check whether your top 10 would have found it.

**What you'll hand in:** a popularity baseline, an item-based collaborative-filtering recommender, content-based similar items, a hybrid with a cold-start rule, hit-rate and coverage results for each, and an explanation feature.

**Skills:** implicit feedback, leave-last-out evaluation, hit rate, item-item collaborative filtering, cosine similarity, TF-IDF, cold start, coverage

**How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**rec_interactions.csv** — one row per purchase: user_id, item_id, day (1–365). Implicit feedback: a purchase means interest; no purchase doesn't mean dislike.

**rec_items.csv** — item_id, category, title (a few words), price (₹)

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return
    print(f"❌ {name}: not yet — {_why(got, want)}")

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
inter = load("rec_interactions.csv")
items = load("rec_items.csv")
K = 10


# Used by the checks: hit rate@K of a recommender over the held-out purchases.
def hit_rate(make_recommender):
    train, test = leave_last_out(inter)
    recommend = make_recommender(train)
    hits = [row.item_id in recommend(row.user_id, K) for row in test.itertuples()]
    return round(float(np.mean(hits)), 3)


def coverage(make_recommender):
    # % of the catalogue that appears in at least one test user's top K
    train, test = leave_last_out(inter)
    rec = make_recommender(train)
    shown = {i for u in test["user_id"] for i in rec(u, K)}
    return round(len(shown) / inter["item_id"].nunique() * 100, 1)
print("Ready! Data loaded.")

## Milestone 1: How sparse is the data?

Write `data_stats(inter)` returning `[sparsity %, median purchases per user, share of purchases going to the top 10% of items (%)]`, each rounded to 1. Sparsity = % of the user × item grid with **no** purchase.

In [ ]:
def data_stats(inter):
    raise NotImplementedError

In [ ]:
check("How sparse is the data?", lambda: data_stats(inter), [85.1, 23.0, 23.2], tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def data_stats(inter):
    users, products = inter["user_id"].nunique(), inter["item_id"].nunique()
    sparsity = (1 - len(inter) / (users * products)) * 100
    per_user = inter.groupby("user_id").size().median()
    counts = inter["item_id"].value_counts()
    top = counts.head(math.ceil(len(counts) * 0.1)).sum() / counts.sum() * 100
    return [round(sparsity, 1), round(float(per_user), 1), round(top, 1)]
```

</details>

## Milestone 2: Hold out each customer's latest purchase

Write `leave_last_out(inter)` returning `(train, test)`: for every user with **at least 5** purchases, their latest purchase (highest `day`; on ties, the last row) goes to `test`; everything else is `train`.

<details>
<summary><b>💡 Hint</b></summary>

Sort by `['user_id', 'day']`, take `groupby('user_id').tail(1)` for users whose count ≥ 5, and drop those rows' index from the data.

</details>

In [ ]:
def leave_last_out(inter):
    raise NotImplementedError

In [ ]:
check("Hold out each customer's latest purchase", lambda: [len(leave_last_out(inter)[0]), len(leave_last_out(inter)[1])], [8556, 388], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def leave_last_out(inter):
    df = inter.sort_values(["user_id", "day"], kind="stable")
    counts = df.groupby("user_id")["item_id"].transform("size")
    test = df[counts >= 5].groupby("user_id").tail(1)
    return df.drop(index=test.index), test
```

</details>

## Milestone 3: Baseline: bestsellers

Write `popular_recommender(train)` that **returns a function** `recommend(user_id, k)`. It recommends the k most-bought items in `train` that this user hasn't bought (ties: smaller item_id first).

The check reports hit rate@10: the share of held-out purchases that appear in the user's top 10.

<details>
<summary><b>💡 Hint</b></summary>

Count once outside the inner function: `order = train.item_id.value_counts()` then sort by (−count, item_id). Inside, skip items in the user's set.

</details>

In [ ]:
def popular_recommender(train):
    def recommend(user_id, k):
        raise NotImplementedError
    return recommend

In [ ]:
check("Baseline: bestsellers", lambda: hit_rate(popular_recommender), 0.198, tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def popular_recommender(train):
    counts = train["item_id"].value_counts()
    order = sorted(counts.index, key=lambda i: (-counts[i], i))
    seen = train.groupby("user_id")["item_id"].apply(set).to_dict()

    def recommend(user_id, k):
        mine = seen.get(user_id, set())
        return [i for i in order if i not in mine][:k]
    return recommend
```

</details>

## Milestone 4: Item-based collaborative filtering

Write `item_cf_recommender(train)` returning `recommend(user_id, k)`:
1. Build a user × item 0/1 matrix from `train` (`pd.crosstab(...).clip(upper=1)`).
2. Item–item similarity = cosine similarity of the item columns.
3. A user's score for item j = Σ similarity(j, i) over items i they bought.
4. Recommend the top k unbought items by score (ties: smaller item_id first).

Does it beat the bestsellers?

<details>
<summary><b>💡 Hint</b></summary>

`S = cosine_similarity(M.T.values)`; scores for a user = `S[:, bought_idx].sum(axis=1)`. Use `np.lexsort((item_ids, -scores))` for a stable tie-break.

</details>

In [ ]:
def item_cf_recommender(train):
    def recommend(user_id, k):
        raise NotImplementedError
    return recommend

In [ ]:
check("Item-based collaborative filtering", lambda: hit_rate(item_cf_recommender), 0.536, tol=0.02)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def item_cf_recommender(train):
    M = pd.crosstab(train["user_id"], train["item_id"]).clip(upper=1)
    ids = M.columns.to_numpy()
    S = cosine_similarity(M.T.values)
    np.fill_diagonal(S, 0)

    def recommend(user_id, k):
        if user_id not in M.index:
            return []
        bought = M.loc[user_id].to_numpy() > 0
        scores = S[:, bought].sum(axis=1)
        order = np.lexsort((ids, -scores))
        return [ids[j] for j in order if not bought[j]][:k]
    return recommend
```

</details>

## Milestone 5: New products: content-based similar items

A new product has no purchases, so collaborative filtering can't recommend it. Use its text instead. Write `similar_items(items, item_id, n=3)` returning the ids of the n most similar items by cosine similarity of **TF-IDF** vectors of `category + " " + title` (excluding the item itself; ties: smaller id first).

In [ ]:
def similar_items(items, item_id, n=3):
    raise NotImplementedError

In [ ]:
check("New products: content-based similar items", lambda: [similar_items(items, 'I007'), similar_items(items, 'I042')], [['I053', 'I145', 'I025'], ['I143', 'I000', 'I052']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def similar_items(items, item_id, n=3):
    text = items["category"] + " " + items["title"]
    X = TfidfVectorizer().fit_transform(text)
    i = items.index[items["item_id"] == item_id][0]
    sims = cosine_similarity(X[i], X).ravel()
    sims[i] = -1
    order = np.lexsort((items["item_id"].to_numpy(), -np.round(sims, 12)))
    return items["item_id"].to_numpy()[order][:n].tolist()
```

</details>

## Milestone 6: A hybrid with a cold-start rule

Customers with few purchases give CF little to work with. Write `hybrid_recommender(train, min_items=6)`: use item-CF for users with at least `min_items` purchases in `train`, and bestsellers otherwise.

The check reports `[hit rate, catalogue coverage %]` where coverage = % of all items that appear in at least one test user's top 10.

In [ ]:
def hybrid_recommender(train, min_items=6):
    def recommend(user_id, k):
        raise NotImplementedError
    return recommend

In [ ]:
check("A hybrid with a cold-start rule", lambda: [hit_rate(hybrid_recommender), coverage(hybrid_recommender)], [0.515, 82.7], tol=0.03)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def hybrid_recommender(train, min_items=6):
    cf, pop = item_cf_recommender(train), popular_recommender(train)
    sizes = train.groupby("user_id").size()

    def recommend(user_id, k):
        return cf(user_id, k) if sizes.get(user_id, 0) >= min_items else pop(user_id, k)
    return recommend
```

</details>

Here the hybrid's hit rate is a little *below* pure item-CF: every test user already has at least 4 purchases, so CF has something to work with. The rule earns its place with truly new customers, who have no history at all. Compare coverage with `coverage(popular_recommender)`: bestsellers show the same few items to everyone. Report trade-offs like this honestly — it's what senior reviewers look for.

## Milestone 7: "Because you bought …"

Explanations build trust. Write `because_you_bought(train, user_id, item_id)` returning the item **this user bought** that is most similar (item-CF cosine) to the recommended `item_id` — that's the item to show in "Because you bought X".

In [ ]:
def because_you_bought(train, user_id, item_id):
    raise NotImplementedError

In [ ]:
check("\"Because you bought \u2026\"", lambda: [because_you_bought(leave_last_out(inter)[0], u, item_cf_recommender(leave_last_out(inter)[0])(u, 1)[0]) for u in ['U001', 'U010', 'U100']], ['I000', 'I115', 'I117'], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def because_you_bought(train, user_id, item_id):
    M = pd.crosstab(train["user_id"], train["item_id"]).clip(upper=1)
    S = pd.DataFrame(cosine_similarity(M.T.values), index=M.columns, columns=M.columns)
    bought = sorted(M.columns[M.loc[user_id] > 0])
    return max(bought, key=lambda i: (round(S.loc[item_id, i], 12), -int(i[1:])))
```

</details>

## Present your work

Write a short design note for the product manager:

1. **Results table**: hit rate@10 and coverage for bestsellers, item-CF and hybrid.
2. **What ships**: the hybrid, and the rule for new customers.
3. **New products**: content-based similar items until they collect purchases.
4. **Explanations**: example "Because you bought" lines for 3 customers.
5. **How to confirm online**: an A/B test on clicks and purchases from the widget.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Built an item-based collaborative-filtering recommender for 400 users × 150 products from implicit purchase data
- Improved hit rate@10 from **X%** (bestsellers) to **Y%** with leave-last-out evaluation, and raised catalogue coverage to **Z%**
- Added TF-IDF similar-item recommendations for cold-start products and "Because you bought" explanations

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. Why evaluate by hiding the latest purchase rather than a random one?
2. What is implicit feedback, and why can't you treat unbought items as dislikes?
3. Item-based vs user-based collaborative filtering — why did you choose item-based?
4. How do you handle a new user? A new product?
5. Hit rate went up. Why might the A/B test still disappoint?
6. How would this scale to 10 million users and 1 million products?

## Take it further

- Add matrix factorisation (implicit ALS or truncated SVD) and compare hit rate.
- Re-rank for diversity: at most 4 items per category in the top 10.
- Weight recent purchases more than old ones.
- Serve recommendations from an API and cache them per user (see project 7).